# Project : Developing a RAG-based Chatbot
---
The following tasks will be undertake :

- **LLM routing**: Develop functions to help categorize and identify the type of each query, creating a router that, depending on the query nature, different treatments are performed.
- **Conditional parameter setting**: Create methods to determine if a user’s query is creative or technical. This allows the LLM to adjust its settings to give the best answer.
- **Producing JSON Responses**: Program the LLMs to generate valid JSON responses with product information, making sure the output is organized for more processing if needed.
- **Adding Contextual Information**: Include relevant data in queries before they are handled by the LLM.
- **Chatbot Development**: Create a chatbot that can interact with users in a natural and efficient way, answering their questions clearly.

# Table of Contents
- [ 1 - Overview](#1)
  - [ 1.1 Imports](#1-1)
  - [ 1.2 Weaviate client setup](#1-2)
- [ 2 - Data schema](#2)
  - [ 2.1 Products database](#2-1)
  - [ 2.2 FAQ database](#2-2)
- [ 3 - Task routing](#3)
  - [ 3.1 Classifying a query as FAQ or product-related](#3-1)
  - [ 3.2 Answering an FAQ question](#3-2)
  - [ 3.3 Identifying the type of product-related question](#3-3)
  - [ 3.4 Extracting task parameters](#3-4)
- [ 4 - Metadata-based retrieval](#4)
  - [ 4.1 Metadata generation](#4-1)
  - [ 4.2 Loading the Weaviate product collection](#4-2)
  - [ 4.3 Filtering by metadata](#4-3)
  - [ 4.4 Building the retrieved items into a context](#4-4)
  - [ 4.5 Querying products](#4-5)
- [ 5 - Putting it together](#5)
  - [ 5.1 The orchestration function](#5-1)
  - [ 5.2 The chatbot](#5-2)

<a id='1'></a>
## 1 - Overview
---

Fashion Forward Hub is a fictional online clothing store used as the setting for this project.
The goal is to build a chatbot for its website, able to answer common questions, provide
product details, and suggest outfits to customers.

The system is built with Retrieval-Augmented Generation (RAG): product and FAQ data is
indexed in a vector store, retrieved at query time, and passed as context to the language
model, so answers stay grounded in the store's actual catalog rather than in the model's
prior knowledge.

<a id='1-1'></a>
### 1.1 Imports

In [ ]:
import json
from weaviate.classes.query import Filter
import weaviate
import joblib

import json
from weaviate.classes.query import Filter
import weaviate
import joblib

import os
import logging
import warnings

# Disable HF specific noise
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HOME"] = "/tmp/huggingface_cache"

# Silence Loggers
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("werkzeug").setLevel(logging.ERROR)

# Silence standard warnings
warnings.filterwarnings("ignore")

In [ ]:
from utils import (
    ChatWidget,
    generate_with_single_input,
    kill_processes_on_ports,
    generate_params_dict
)

# Kill processes on ports before importing flask_app and weaviate_server.
# kill_processes_on_ports is kept in utils.py specifically for this step.
# WARNING: Running this cell twice may kill the active kernel
kill_processes_on_ports([5000, 8080, 8097, 50050, 50051])

import flask_app
import weaviate_server

<a id='1-2'></a>
### 1.2 Weaviate client setup

The product and FAQ collections are stored in a Weaviate vector database. This section
opens a client connection to the running instance; the collections are populated
and are queried read-only throughout the project.

In [ ]:
client = weaviate.connect_to_local(port=8079, grpc_port=50050)

## 1.3 The `generate_params_dict` helper

A small helper builds the parameter dictionary passed to the completion API, so that
model, sampling settings and message role are declared in one place instead of being
repeated at every call site:

```Python
def generate_params_dict(
    prompt: str, 
    temperature: float = None, 
    role = 'user',
    top_p: float = None,
    max_tokens: int = 500,
    model: str = "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo"
)
```

In [4]:
# An output example is
kwargs = generate_params_dict("Solve x^2 - 1 = 0", temperature = 1.2, top_p = 0.2)
print(kwargs)

{'prompt': 'Solve x^2 - 1 = 0', 'role': 'user', 'temperature': 1.2, 'top_p': 0.2, 'max_tokens': 500, 'model': 'Qwen/Qwen3.5-9B'}


In [4]:
# Generating 
response = generate_with_single_input(**kwargs)
print(response['content'])

To solve the equation $x^2 - 1 = 0$, we can follow these steps:

### Step-by-Step Deduction

**Method 1: Factoring (Difference of Squares)**

1.  Recognize that the expression $x^2 - 1$ is a difference of two squares, which follows the pattern $a^2 - b^2 = (a - b)(a + b)$.
2.  Rewrite the equation:
    $$x^2 - 1^2 = 0$$
3.  Factor the left side:
    $$(x - 1)(x + 1) = 0$$
4.  Apply the Zero Product Property, which states that if the product of two factors is zero, then at least one of the factors must be zero. Therefore:
    $$x - 1 = 0 \quad \text{or} \quad x + 1 = 0$$
5.  Solve for $x$ in each case:
    *   $x - 1 = 0 \implies x = 1$
    *   $x + 1 = 0 \implies x = -1$

**Method 2: Isolating the Variable**

1.  Add $1$ to both sides of the equation:
    $$x^2 = 1$$
2.  Take the square root of both sides. Remember that taking the square root of a positive number yields both a positive and a negative result:
    $$x = \pm\sqrt{1}$$
3.  Simplify the square root:
    $$x = \pm 1$$

### F

<a id='2'></a>
## 2 - Data schema
---
The application reads from two data sources:

- **Product database** — the catalog items and their attributes.
- **FAQ database** — the store's frequently asked questions.

<a id='2-1'></a>
### 2.1 Products database

The product catalog is loaded here as a list of JSON objects, which makes the field
structure easier to inspect before it is indexed.

In [5]:
# Loading products data
PRODUCTS_DATA = joblib.load('dataset/clothes_json.joblib')

In [6]:
# Let's get one example
PRODUCTS_DATA[0]

{'gender': 'Men',
 'masterCategory': 'Apparel',
 'subCategory': 'Topwear',
 'articleType': 'Shirts',
 'baseColour': 'Navy Blue',
 'season': 'Fall',
 'year': 2011.0,
 'usage': 'Casual',
 'productDisplayName': 'Turtle Check Men Navy Blue Shirt',
 'price': 67,
 'product_id': 15970}

The features each product has are:

- **Gender:** Target audience for the product, such as "Men," "Women," or "Unisex."
- **Master Category:** Broad classification like "Apparel" or "Footwear."
- **Sub Category:** Specific category within a master category, such as "Topwear."
- **Article Type:** Exact type of product, e.g., "Shirts" or "Jackets."
- **Base Colour:** Main color of the product, important for customer choice.
- **Season:** Intended season for the product, e.g., "Summer" or "Winter."
- **Year:** Year of release or collection.
- **Usage:** Intended use or occasion, like "Casual" or "Formal."
- **Product Display Name:** Descriptive name used in marketing.
- **Price:** Cost of the product.
- **Product ID:** Unique identifier for managing and tracking inventory.

<a id='2-2'></a>
### 2.2 FAQ Database

Now let's load the FAQ database. And explore it.

In [7]:
FAQ = joblib.load("dataset/faq.joblib")

In [8]:
# Get an example
FAQ[:2]

[{'question': 'What are your store hours?',
  'answer': 'Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday.',
  'type': 'general information'},
 {'question': 'Where is Fashion Forward Hub located?',
  'answer': 'Fashion Forward Hub is primarily an online store. Our corporate office is located at 123 Fashion Lane, Trend City, Style State.',
  'type': 'general information'}]

Each FAQ entry is a dictionary with `question`, `answer` and `type` fields. The FAQ set is
small enough to be injected directly into the prompt as a string, so it is not indexed as a
separate vector collection — retrieval is only needed on the product side.

<a id='3'></a>
## 3 - Task routing
---
<a id='3-1'></a>
### 3.1 Classifying a query as FAQ or product-related

### 3.1 Classifying a query as FAQ or product-related

The router is the entry point of the pipeline. It reads the user's query and decides which
of the two branches handles it: FAQ questions are answered from the static FAQ set, while
product queries go through metadata extraction and vector retrieval. Product queries cover
both factual lookups ("blue t-shirts under $100") and open-ended requests, such as putting
together a look for a given occasion.

Classification is done with a single LLM call using a few-shot prompt: the two labels are named explicitly, two examples illustrate the boundary, and sampling is kept tight (`temperature=0.3`, `top_p=0.2`) with a hard cap on output length.

In [ ]:
def check_if_faq_or_product(query: str) -> str | None:
    """
    Route a user query to one of the two branches of the pipeline.

    Parameters:
    - query (str): the raw user query.

    Returns:
    - str | None: 'Product' for queries about catalog items, 'FAQ' for queries about
      store policies, or None if the model returned anything else.
    """
    prompt = f"""As an assistant working in an online clothing store, classify the given query in two classes: Product or FAQ, returning exactly one word.
    A query labeled Product refers to information related to specific product details, generally focusing on product attributes such as price, gender and article type.
    A query labeled FAQ refers to questions users may have about the company and its policies.

    Example:
    query: Do you have this shirt in another size?
    returns: Product

    query: Is there any available discount for students?
    returns: FAQ

    Classify this query: {query}"""

    kwargs = generate_params_dict(prompt, temperature=0.3, top_p=0.2, max_tokens=1)
    response = generate_with_single_input(**kwargs)

    label = response["content"].strip()
    return label if label in ("Product", "FAQ") else None

In [9]:
queries = ['What is your return policy?', 
           'Give me three examples of blue T-shirts you have available.', 
           'How can I contact the user support?', 
           'Do you have blue Dresses?',
           'Create a look suitable for a wedding party happening during dawn.']

for query in queries:
    response = check_if_faq_or_product(query)
    label = response
    print(f"Query: {query} Label: {label}")

Query: What is your return policy? Label: FAQ
Query: Give me three examples of blue T-shirts you have available. Label: Product
Query: How can I contact the user support? Label: FAQ
Query: Do you have blue Dresses? Label: Product
Query: Create a look suitable for a wedding party happening during dawn. Label: Product


<a id='3-2'></a>
### 3.2 Answering an FAQ question

Queries routed to the FAQ branch are answered from the question/answer pairs of the FAQ
database. Since the set is small, the whole thing is flattened into a single string and
injected into the prompt, so the model answers from the store's actual policies rather than
from its own priors.

In [12]:
# print the structure of the first element
FAQ[0]

{'question': 'What are your store hours?',
 'answer': 'Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday.',
 'type': 'general information'}

#### 3.2.1 Building the FAQ layout

The FAQ entries are serialized one per line, in the form:

```
Question: FAQ Question 1 Answer: FAQ Answer 1 Type: FAQ Type 1
...
Question: FAQ Question 25 Answer: FAQ Answer 25 Type: FAQ Type 25
```

In [ ]:
def generate_faq_layout(faq_entries: list[dict]) -> str:
    """
    Serialize FAQ entries into a single string for prompt injection.

    Parameters:
    - faq_entries (list[dict]): FAQ entries, each with 'question', 'answer' and 'type' keys.

    Returns:
    - str: one entry per line, in the form "Question: ... Answer: ... Type: ...".
    """
    return "".join(
        f"Question: {entry['question']} Answer: {entry['answer']} Type: {entry['type']}\n"
        for entry in faq_entries
    )

In [12]:
FAQ_LAYOUT = generate_faq_layout(FAQ)
print(FAQ_LAYOUT[:1000])

Question: What are your store hours? Answer: Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday. Type: general information
Question: Where is Fashion Forward Hub located? Answer: Fashion Forward Hub is primarily an online store. Our corporate office is located at 123 Fashion Lane, Trend City, Style State. Type: general information
Question: Do you have a physical store location? Answer: At this time, we operate exclusively online. This allows us to offer a broader selection and lower prices directly to you. Type: general information
Question: How can I create an account with Fashion Forward Hub? Answer: Click on 'Sign Up' in the top right corner of our website and follow the instructions to set up your account. Type: general information
Question: How do I subscribe to your newsletter? Answer: To receive the latest updates and promotions, sign up for our newsletter at the bottom of our homepage. Type: general information
Question:

The FAQ layout is wrapped in `<FAQ>` tags to mark the context boundary. The prompt allows
the model to combine several entries when one alone doesn't cover the question, and forbids
it from mentioning the FAQ itself, so the reply reads as a direct answer from the store.

In [ ]:
def query_on_faq(query: str, **kwargs) -> dict:
    """
    Build the LLM call parameters for answering a query from the FAQ content.

    Parameters:
    - query (str): the user query, already routed to the FAQ branch.
    - **kwargs: optional overrides forwarded to generate_params_dict (temperature, model...).

    Returns:
    - dict: the parameter dictionary to pass to generate_with_single_input.
    """
    prompt = f"""You will be provided with an FAQ for a clothing store. Answer the instruction based on it. You might use more than one question and answer to make your answer.
    Only answer the question and do not mention that you have access to a FAQ.
    <FAQ>{FAQ_LAYOUT}</FAQ>
    Question: {query}"""

    return generate_params_dict(prompt, **kwargs)

In [16]:
kwargs = query_on_faq("I got my cloth but I didn't like it. How can I return it?")

In [17]:
content = generate_with_single_input(**kwargs)

In [18]:
print(content['content'])

To return an item, initiate a return through the Returns Center on the website, selecting the item you wish to exchange or return. You must start the process within 30 days of delivery. If returning internationally, shipping costs are at your expense; for domestic returns, a prepaid return label will be provided. Please note that sale items are typically final sale and cannot be returned unless stated otherwise.


<a id='3-3'></a>
### 3.3 Identifying the type of product-related question

Queries routed to the product branch are classified a second time, into two kinds:

- **Technical** — asking for specific catalog information: whether a blue dress is in stock,
  three red t-shirts suitable for warm weather, items under a given price.
- **Creative** — asking the assistant to compose something: a look for a museum visit, an
  outfit for a wedding.

The distinction matters downstream: technical queries map fairly directly onto metadata
filters, while creative ones need the model to invent a coherent set of attributes before
anything can be retrieved. Classification uses the same single-token, few-shot approach as
the FAQ/Product router.

In [ ]:
def decide_task_nature(query: str) -> str | None:
    """
    Classify a product-related query as creative or technical.

    Parameters:
    - query (str): the user query, already routed to the product branch.

    Returns:
    - str | None: 'creative' if the query asks the assistant to compose something,
      'technical' if it asks for catalog information, or None if the model returned
      anything else.
    """
    prompt = f"""Decide if the following query requires creativity (creating, composing, making new things) or is technical (information about products, prices, etc.).
    Label it as creative or technical.
    Examples:
    Give me suggestions on a nice look for a nightclub. Label: creative
    What are the blue dresses you have available? Label: technical
    Give me three T-shirts for summer. Label: technical
    Give me a look for attending a wedding party. Label: creative

    Query to be analyzed: {query}. Only output one token: the label."""

    kwargs = generate_params_dict(prompt, temperature=0, max_tokens=1)
    response = generate_with_single_input(**kwargs)

    label = response["content"].strip().lower()
    return label if label in ("creative", "technical") else None

In [16]:
queries = ["Give me two sneakers with vibrant colors.",
           "What are the most expensive clothes you have in your catalogue?",
           "I have a green dress and I like a suggestion on an accessory to match with it.",
           "Give me three trousers with vibrant colors you have in your catalogue.",
           "Create a look for a woman walking in a park on a sunny day. It must be fresh due to hot weather."
           ]

In [17]:
for query in queries:
    label = decide_task_nature(query)
    print(f"Query: {query} Label: {label}")

Query: Give me two sneakers with vibrant colors. Label: technical
Query: What are the most expensive clothes you have in your catalogue? Label: technical
Query: I have a green dress and I like a suggestion on an accessory to match with it. Label: creative
Query: Give me three trousers with vibrant colors you have in your catalogue. Label: technical
Query: Create a look for a woman walking in a park on a sunny day. It must be fresh due to hot weather. Label: creative


<a id='3-4'></a>
### 3.4 Sampling parameters per task type

The two task types don't want the same decoding behaviour. Technical answers must stay
close to the retrieved catalog data, so sampling is kept tight; creative answers benefit
from more variety, since the point is to propose a combination the user hadn't thought of.

Each label maps to its own `temperature` / `top_p` pair, with a conservative default used
when the classifier returns an unexpected value — an invalid label means the query type is
unknown, and inventing freely on an unknown query is the worse failure mode.

In [ ]:
# Sampling profiles per task type. Tuned so that technical answers stay close to the
# retrieved data, while creative ones can explore more combinations.
SAMPLING_PROFILES = {
    "technical": {"temperature": 0.3, "top_p": 0.5},
    "creative": {"temperature": 1.0, "top_p": 0.3},
}
DEFAULT_SAMPLING = {"temperature": 0.6, "top_p": 0.4}


def get_params_for_task(task: str) -> dict:
    """
    Return the sampling parameters matching a task type.

    Parameters:
    - task (str): 'creative' or 'technical'.

    Returns:
    - dict: 'temperature' and 'top_p' for that task, or a conservative default
      if the task type is not recognized.
    """
    return SAMPLING_PROFILES.get(task, DEFAULT_SAMPLING)

In [20]:
get_params_for_task("technical")

{'top_p': 0.5, 'temperature': 0.3}

<a id='4'></a>
## 4 - Metadata-based retrieval
---
Product queries are answered by first inferring structured metadata from the natural
language query, then using it to filter the vector search. A JSON file lists every field
and the distinct values it takes in the dataset; this vocabulary is injected into the
prompt so the model selects from existing values rather than inventing labels the database
has never seen. Values that still fall outside the vocabulary are discarded downstream.

Six fields are used:

- `gender`
- `masterCategory`
- `articleType`
- `baseColour`
- `season`
- `usage`

They are specific enough to narrow the catalog meaningfully, while staying coarse enough
that a filter rarely returns nothing. Finer-grained fields in the dataset were left out for
that reason. Field count also has a direct cost: every additional field brings its whole
value list into the prompt, inflating both latency and token spend.

In [22]:
# Let's remember the data structure of a product
PRODUCTS_DATA[0]

{'gender': 'Men',
 'masterCategory': 'Apparel',
 'subCategory': 'Topwear',
 'articleType': 'Shirts',
 'baseColour': 'Navy Blue',
 'season': 'Fall',
 'year': 2011.0,
 'usage': 'Casual',
 'productDisplayName': 'Turtle Check Men Navy Blue Shirt',
 'price': 67,
 'product_id': 15970}

In [23]:
# Run this cell to generate the dictionary with the possible values for each key
values = {}
for d in PRODUCTS_DATA:
    for key, val in d.items():
        if key in ('product_id', 'price', 'productDisplayName', 'subCategory', 'year'):
            continue
        if key not in values.keys():
            values[key] = set()
        values[key].add(val)

In [24]:
# Example of possible values for the feature 'season'
values['season']

{'All seasons', 'Fall', 'Spring', 'Summer', 'Winter'}

<a id='4-1'></a>
### 4.1 Metadata generation

This step turns a natural language query into a structured filter. The dataset vocabulary
is injected into the prompt so the model picks from values the database actually contains,
and the output is a JSON object with one key per filterable field.

Price is handled alongside the categorical fields. When the query states a range, it is
returned as bounds; when it doesn't, the field falls back to an unbounded interval:

```json
"price": {"min": 0, "max": "inf"}
```

The expected output shape:

```json
{
    "gender": ["Women"],
    "masterCategory": ["Apparel"],
    "articleType": ["Dresses"],
    "baseColour": ["Blue"],
    "price": {"min": 0, "max": "inf"},
    "usage": ["Formal"],
    "season": ["All seasons"]
}
```

`max_tokens` is set to 1500: the value lists injected in the prompt are long, and a tighter
budget truncates the response mid-object, which then fails to parse.

In [ ]:
def generate_metadata_from_query(query: str) -> str:
    """
    Infer structured filter metadata from a natural language product query.

    Parameters:
    - query (str): the user query, already routed to the product branch.

    Returns:
    - str: a JSON string with the keys gender, masterCategory, articleType, baseColour,
      usage and season (each a list of values taken from the dataset vocabulary), plus a
      price key holding "min" and "max" bounds — 0 and "inf" when the query sets none.
    """
    prompt = f"""Generate metadata in JSON format to filter clothing items for a given query.
    The possible values for each feature are given in the following JSON: {values}
    Only use values present in that JSON. If no value fits a feature, return an empty list for it.

    Always include gender, masterCategory, articleType, baseColour, price, usage and season as keys.
    All values must be inside lists, except price, which is an object with "min" and "max"
    (use 0 when there is no lower bound and "inf" when there is no upper bound).
    Return the JSON only, with no surrounding text and no markdown code fences.

    Example:

    query: Is this blue dress still available?
    returns: {{
    "gender": ["Women"],
    "masterCategory": ["Apparel"],
    "articleType": ["Dresses"],
    "baseColour": ["Blue"],
    "price": {{"min": 0, "max": "inf"}},
    "usage": ["Formal"],
    "season": ["All seasons"]
    }}

    Query: {query}
    """

    kwargs = generate_params_dict(prompt, temperature=0, max_tokens=1500)
    response = generate_with_single_input(**kwargs)

    return response["content"].strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

In [72]:
print(generate_metadata_from_query("Create a look for a man that suits a sunny day in the park. I don't want to spend more than 300 dollars on each piece."))

{
  "gender": ["Men"],
  "masterCategory": ["Apparel"],
  "articleType": ["Tshirts", "Shorts", "Casual Shoes", "Sunglasses", "Caps"],
  "baseColour": ["White", "Beige", "Khaki", "Navy Blue", "Grey", "Black", "Blue", "Green"],
  "price": {"min": 0, "max": 300},
  "usage": ["Casual", "All occasions"],
  "season": ["Summer", "All seasons"]
}


The next helpers extract the JSON object from the model's raw output. Since generation can
produce malformed or unparseable JSON, failures are caught and turned into an empty filter
rather than an exception — an over-broad search is a better outcome than a crashed request.

In [ ]:
def parse_json_output(llm_output: str) -> dict | None:
    """
    Parse the model's raw output into a dictionary.

    The output may carry minor formatting artifacts (stray newlines, doubled braces echoed
    from the prompt example), which are cleaned before parsing.

    Parameters:
    - llm_output (str): the raw text returned by the model.

    Returns:
    - dict | None: the parsed object, or None if the text is not valid JSON.
    """
    cleaned = llm_output.replace("\n", "").replace("{{", "{").replace("}}", "}")

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        return None

In [28]:
json_string = generate_metadata_from_query("Give me three blue dresses suitable for a wedding party, less than 200 dollars and at least 50 dollars")
json_output = parse_json_output(json_string)

In [29]:
json_output

{'gender': ['Women'],
 'masterCategory': ['Apparel'],
 'articleType': ['Dresses'],
 'baseColour': ['Blue'],
 'price': {'min': 50, 'max': 200},
 'usage': ['Party'],
 'season': ['All seasons']}

<a id='4-2'></a>
### 4.2 Loading the Weaviate product collection

The product catalog seen above is also stored as a Weaviate collection, which allows the
same items to be reached by semantic search and narrowed by metadata filters in a single
query.

In [30]:
products_collection = client.collections.get('products')

In [31]:
len(products_collection)

44423

<a id='4-3'></a>
### 4.3 Filtering by metadata

The metadata inferred from the query is translated into Weaviate filters — one `Filter`
object per field, combined into a list. Categorical fields match on any of the proposed
values; price is handled separately, as a numeric range rather than a set membership.

In [ ]:
def get_filter_by_metadata(json_output: dict | None = None) -> list | None:
    """
    Translate inferred metadata into a list of Weaviate filters.

    Parameters:
    - json_output (dict | None): metadata keys and their values, as returned by
      parse_json_output.

    Returns:
    - list[Filter] | None: one filter per usable field, or None if no metadata was parsed.
    """
    if json_output is None:
        return None

    valid_keys = (
        "gender",
        "masterCategory",
        "articleType",
        "baseColour",
        "price",
        "usage",
        "season",
    )

    filters = []

    for key, value in json_output.items():
        if key not in valid_keys:
            continue

        if key == "price":
            if not isinstance(value, dict):
                continue

            min_price = value.get("min")
            max_price = value.get("max")

            if min_price is not None and min_price > 0:
                filters.append(Filter.by_property(key).greater_than(min_price))
            if max_price is not None and max_price != "inf":
                filters.append(Filter.by_property(key).less_than(max_price))
        else:
            filters.append(Filter.by_property(key).contains_any(value))

    return filters

This is wrapper function, that, given a query, return the desired filters.

In [34]:
def generate_filters_from_query(query: str) -> list:
    json_string = generate_metadata_from_query(query)
    json_output = parse_json_output(json_string)
    filters = get_filter_by_metadata(json_output)
    return filters

In [35]:
filters = generate_filters_from_query("Give me three T-shirts to use in sunny days")

In [36]:
filters

[_FilterValue(value=['Unisex', 'Men', 'Women'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='gender'),
 _FilterValue(value=['Apparel'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='masterCategory'),
 _FilterValue(value=['Tshirts', 'Lounge Tshirts'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='articleType'),
 _FilterValue(value=['White', 'Black', 'Grey', 'Blue', 'Red', 'Green', 'Yellow', 'Orange', 'Pink', 'Purple', 'Multi'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='baseColour'),
 _FilterValue(value=['Casual', 'Sports'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='usage'),
 _FilterValue(value=['Summer', 'All seasons'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='season')]

The next function performs the actual retrieval: it builds the filters, runs a semantic
search on the query text, and applies the filters to narrow the candidates.

Strict filtering on six fields often matches very few items, so filters are dropped
progressively — least discriminating first — until the result set is large enough. If no
combination works, the search falls back to pure semantic similarity with no filters at all.

In [ ]:
# Filters are dropped in this order when the result set is too small. Price and
# articleType are never dropped: they carry the user's explicit intent.
RELAXATION_ORDER = ("season", "usage", "baseColour", "masterCategory", "gender")
MIN_RESULTS = 5
SEARCH_LIMIT = 20


def get_relevant_products_from_query(query: str) -> list:
    """
    Retrieve the catalog items most relevant to a query.

    Combines semantic search on the query text with metadata filters inferred from it.
    When the filtered search returns too few items, filters are relaxed step by step.

    Parameters:
    - query (str): the user query.

    Returns:
    - list: the retrieved product objects, at most SEARCH_LIMIT of them.
    """
    filters = generate_filters_from_query(query)

    if not filters:
        return products_collection.query.near_text(query, limit=SEARCH_LIMIT).objects

    res = products_collection.query.near_text(
        query, filters=Filter.all_of([f for _, f in filters]), limit=SEARCH_LIMIT
    ).objects
    if len(res) >= MIN_RESULTS:
        return res

    for depth in range(1, len(RELAXATION_ORDER) + 1):
        dropped = RELAXATION_ORDER[:depth]
        kept = [f for name, f in filters if name not in dropped]
        if not kept:
            break

        res = products_collection.query.near_text(
            query, filters=Filter.all_of(kept), limit=SEARCH_LIMIT
        ).objects
        if len(res) >= MIN_RESULTS:
            return res

    return products_collection.query.near_text(query, limit=SEARCH_LIMIT).objects

In [39]:
query = "Give me three T-shirts to use in sunny days"

In [40]:
t = get_relevant_products_from_query("Give me three T-shirts to use in sunny days")

In [ ]:
if len(t) > 0:
    print(t[0].properties)

{'subCategory': 'Topwear', 'gender': 'Men', 'productDisplayName': 'ADIDAS MenLime Green T-shirt', 'baseColour': 'Fluorescent Green', 'masterCategory': 'Apparel', 'articleType': 'Tshirts', 'season': 'Summer', 'usage': 'Casual', 'product_id': 40093, 'price': 190, 'year': 2012}


So, one of the relevant results is indeed a Tshirt! 

<a id='4-4'></a>
### 4.4 Building the retrieved items into a context

The retrieved objects are flattened into plain sentences, one per item, which are then
appended to the generation prompt:

```
Product name: Inkfruit Mens Little Bit More T-shirt. Product Category: Apparel. Product usage: Casual. Product gender: Men. Product Type: Tshirts. Product Category: Topwear Product Color: Yellow. Product Season: Summer. Product Year: 2011.
```

In [ ]:
def generate_items_context(results: list) -> str:
    """
    Flatten retrieved product objects into a plain-text context block.

    Parameters:
    - results (list): objects returned by the Weaviate query, each exposing a
      `properties` mapping.

    Returns:
    - str: one line per product, listing the attributes passed to the model.
    """
    lines = []

    for item in results:
        p = item.properties
        lines.append(
            f"Product ID: {p['product_id']}. "
            f"Product name: {p['productDisplayName']}. "
            f"Product Category: {p['masterCategory']}. "
            f"Product Subcategory: {p['subCategory']}. "
            f"Product Type: {p['articleType']}. "
            f"Product usage: {p['usage']}. "
            f"Product gender: {p['gender']}. "
            f"Product Color: {p['baseColour']}. "
            f"Product Season: {p['season']}. "
            f"Product Year: {p['year']}."
        )

    return "\n".join(lines) + "\n" if lines else ""

In [43]:
print(generate_items_context(t)[:1000])

Product ID: 40093. Product name: ADIDAS MenLime Green T-shirt. Product Category: Apparel. Product usage: Casual. Product gender: Men. Product Type: Tshirts. Product Category: Topwear Product Color: Fluorescent Green. Product Season: Summer. Product Year: 2012.
Product ID: 1844. Product name: Inkfruit Mens D day T-shirt. Product Category: Apparel. Product usage: Casual. Product gender: Men. Product Type: Tshirts. Product Category: Topwear Product Color: Blue. Product Season: Summer. Product Year: 2011.
Product ID: 1853. Product name: Inkfruit Mens Little Bit More T-shirt. Product Category: Apparel. Product usage: Casual. Product gender: Men. Product Type: Tshirts. Product Category: Topwear Product Color: Yellow. Product Season: Summer. Product Year: 2011.
Product ID: 1847. Product name: Inkfruit Mens Messy T-shirt. Product Category: Apparel. Product usage: Casual. Product gender: Men. Product Type: Tshirts. Product Category: Topwear Product Color: Blue. Product Season: Summer. Product Y

<a id='4-5'></a>
### 4.5 Querying products

The product branch chains the pieces together:

1. Classify the query as technical or creative.
2. Select the sampling profile matching that type.
3. Retrieve the relevant catalog items, with metadata filtering and relaxation.
4. Flatten them into a context block.
5. Build the generation prompt from the context and the query.
6. Return the call parameters.

In [ ]:
def query_on_products(query: str) -> dict:
    """
    Build the LLM call parameters for answering a product query.

    Retrieves catalog items matching the query, formats them as context, and pairs the
    resulting prompt with the sampling profile of the query type.

    Parameters:
    - query (str): the user query, already routed to the product branch.

    Returns:
    - dict: the parameter dictionary to pass to generate_with_single_input.
    """
    query_label = decide_task_nature(query)
    parameters_dict = get_params_for_task(query_label)

    relevant_products = get_relevant_products_from_query(query)
    context = generate_items_context(relevant_products)

    prompt = (
        "Given the available set of clothing products, answer the question that follows, providing the item ID in your answers. "
        "Not all attributes are relevant to every query; pick only the ones that matter and keep the descriptions short. "
        "If the query does not state how many products to return, show at most five. "
        f"CLOTH PRODUCTS AVAILABLE: {context} "
        f"QUERY: {query}"
    )

    return generate_params_dict(prompt, role="assistant", **parameters_dict)

In [50]:
kwargs = query_on_products('Make a wonderful look for a man attending a wedding party happening during night.')

In [51]:
result = generate_with_single_input(**kwargs)
print(result['content'])

For a night wedding, a black tie is the most appropriate choice for a formal look. Here are the relevant product IDs from the available set:

*   **40206** (Provogue Men Black Tie)
*   **40234** (Provogue Men Black Tie)
*   **40210** (Provogue Men Black Tie)
*   **50284** (Parx Men Black Tie)
*   **50286** (Parx Men Black Tie)

*(Note: All listed items are black, formal men's ties suitable for evening events.)*


In [52]:
kwargs = query_on_products('Give me three T-shirts for sunny days')

In [53]:
result = generate_with_single_input(**kwargs)
print(result['content'])

Based on the query for T-shirts suitable for sunny days (Summer season), here are three relevant items:

*   **40093**: ADIDAS MenLime Green T-shirt (Fluorescent Green, Summer)
*   **1844**: Inkfruit Mens D day T-shirt (Blue, Summer)
*   **1853**: Inkfruit Mens Little Bit More T-shirt (Yellow, Summer)


<a id='5'></a>
## 5 - Putting it together
---
<a id='5-1'></a>
### 5.1 The orchestration function

A single entry point routes the query and runs the matching workflow, returning the call
parameters in both cases. Queries that fit neither branch, and failures inside the product
pipeline, fall back to a prompt that lets the assistant answer from the conversation
context alone.

In [ ]:
def answer_query(query: str) -> dict:
    """
    Route a query to the FAQ or product workflow and return the LLM call parameters.

    Parameters:
    - query (str): the raw user query.

    Returns:
    - dict: the parameter dictionary to pass to generate_with_single_input. Unroutable
      queries and product-pipeline failures return a fallback prompt instead.
    """
    label = check_if_faq_or_product(query)

    if label == "FAQ":
        return query_on_faq(query)

    if label == "Product":
        try:
            return query_on_products(query)
        except Exception:
            logging.exception("Product pipeline failed for query: %s", query)
            return generate_params_dict(
                "The product lookup failed for this query. Answer from the context you already have, "
                f"and ask the user to rephrase their request. Query: {query}",
                role="assistant",
            )

    return generate_params_dict(
        "This query fits neither the FAQ nor the product catalog. Answer it from the context you already have. "
        f"Query: {query}",
        role="assistant",
    )

In [55]:
kwargs = answer_query("What are your working hours?")

In [56]:
result = generate_with_single_input(**kwargs)
print(result['content'])

The online store is open 24/7, and customer service is available from 9:00 AM to 6:00 PM, Monday through Friday.


<a id='5-2'></a>
### 5.2 The chatbot

The conversational loop lives in `utils.py`: it keeps the message history, calls
`answer_query` on each turn, and passes the resulting parameters to the model.

Example queries:

- Do you have blue t-shirts in your catalogue?
- I bought a dress and I didn't like it. How can I get a refund?
- I'm going to a beach party. Can you suggest a nice look for me? It will be a warm night, and I'm a man.

In [ ]:
chat_widget = ChatWidget(generator_function = answer_query)

## Limitations and next steps

- The FAQ set is injected into the prompt rather than indexed; this stops scaling once
  the FAQ grows beyond a few dozen entries.
- Classification relies on two chained LLM calls, which adds latency to every turn. A
  single call returning both labels, or a small local classifier, would cut it.
- Filter relaxation drops fields in a fixed order rather than by measured selectivity on
  the actual catalog.
- No evaluation set: routing accuracy and retrieval quality are only checked by hand.